# Customer Churn Prediction with MLflow

## Import Libraries

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from mlflow.tracking import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, f1_score, classification_report

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 3, Finished, Available, Finished, False)

## Read the dataset

**Objective**: Read the CSV dataset into a Pandas DataFrame and inspect its structure.

In [2]:
DATA_PATH = "/lakehouse/default/Files/churn.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Number of records:", len(df))

display(df.head())

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 5, Finished, Available, Finished, False)

Dataset loaded successfully
Number of records: 5000


SynapseWidget(Synapse.DataFrame, 1150e9d0-8f39-478f-af05-a9b9a2041205)

# Model Building

## Prepare features and target

**Objective**: Separate the input variables from the target variable.

In [3]:
FEATURE_COLUMNS = [
    "tenure_months",
    "monthly_charges",
    "support_calls",
    "contract_type"
]

TARGET_COLUMN = "churn"

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

print("Features:", FEATURE_COLUMNS)
print("Target:", TARGET_COLUMN)

print("\nTarget distribution:")
print(y.value_counts())

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 7, Finished, Available, Finished, False)

Features: ['tenure_months', 'monthly_charges', 'support_calls', 'contract_type']
Target: churn

Target distribution:
churn
0    2724
1    2276
Name: count, dtype: int64


## Prepare training and testing data

**Objective:** Create separate datasets for model training, hyperparameter selection, and final testing.

Use a 60:20:20 split.

We will use:
- 60% for training
- 20% for validation
- 20% for final testing

In [4]:
# Split 80% development data and 20% test data
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split development data into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training records:", len(X_train))
print("Validation records:", len(X_val))
print("Testing records:", len(X_test))

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 8, Finished, Available, Finished, False)

Training records: 3000
Validation records: 1000
Testing records: 1000


# ML Experiment

## Configure MLflow experiment

**Objective**: Create an MLflow experiment to track the model training runs.

* We will disable automatic logging to avoid unnecessary metrics and artifacts.

In [5]:
mlflow.sklearn.autolog(disable=True)

EXPERIMENT_NAME = "EXPERIMENT_CustomerChurn_Tuning"

mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow Experiment:", EXPERIMENT_NAME)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 9, Finished, Available, Finished, False)

2026-11-10 19:26:35 [INFO] setting mlflow tracking uri to sds://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow, change this environment variable if you want log to other places
MLflow Experiment: EXPERIMENT_CustomerChurn_Tuning


2026/11/10 19:26:45 INFO mlflow.tracking.fluent: Experiment with name 'EXPERIMENT_CustomerChurn_Tuning' does not exist. Creating a new experiment.


## Train models and track experiments

**Objective**: Train Logistic Regression models using different regularization values and track their performance.


<u>Concept to understand – Regularization parameter C</u>
- A smaller C value applies stronger regularization.
- A larger C value applies weaker regularization.
- The best value depends on the dataset and model performance.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

C_VALUES = [0.0001, 0.001, 0.01, 0.1]

results = []

for c_value in C_VALUES:

    with mlflow.start_run(
        run_name=f"LogisticRegression_C_{c_value}"
    ):

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=c_value,
                max_iter=1000,
                solver="lbfgs"
            ))
        ])

        model.fit(X_train, y_train)

        val_predictions = model.predict(X_val)

        val_accuracy = accuracy_score(
            y_val,
            val_predictions
        )

        val_f1 = f1_score(
            y_val,
            val_predictions
        )

        mlflow.log_param(
            "algorithm",
            "LogisticRegression"
        )

        mlflow.log_param(
            "regularization_C",
            c_value
        )

        mlflow.log_metric(
            "validation_accuracy",
            val_accuracy
        )

        mlflow.log_metric(
            "validation_f1_score",
            val_f1
        )

        results.append({
            "C": c_value,
            "validation_accuracy": val_accuracy,
            "validation_f1_score": val_f1
        })

        print(
            f"C={c_value} | "
            f"Accuracy={val_accuracy:.4f} | "
            f"F1={val_f1:.4f}"
        )

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 10, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C=0.0001 | Accuracy=0.5620 | F1=0.0720
🏃 View run LogisticRegression_C_0.0001 at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f/runs/f79e7568-6ea6-42e9-8819-f9576e1497b6
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f


C=0.001 | Accuracy=0.7960 | F1=0.7524
🏃 View run LogisticRegression_C_0.001 at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f/runs/e710f088-16e6-436d-8a22-8cce0b717f60
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f


C=0.01 | Accuracy=0.8200 | F1=0.7968
🏃 View run LogisticRegression_C_0.01 at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f/runs/42ae1116-4bf7-42ae-9e81-bd1d6233a7ee
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f


C=0.1 | Accuracy=0.8210 | F1=0.8004
🏃 View run LogisticRegression_C_0.1 at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f/runs/51dd63d6-cf0e-4169-9896-b7126d09d2e3
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f


## Compare experiment results

**Objective**: Compare the validation performance of all model runs and identify the best hyperparameter.

In [7]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="validation_accuracy",
    ascending=False
)

display(results_df)

BEST_C = results_df.iloc[0]["C"]

print("Best regularization value:", BEST_C)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e68677c0-192f-434a-b9f6-26bd0df67772)

Best regularization value: 0.1


# Identify Best Model

## Using Experient UI

## Using Code

### Identify Model

In [8]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="validation_accuracy",
    ascending=False
)

display(results_df)

BEST_C = results_df.iloc[0]["C"]

print("Best C value:", BEST_C)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b5d51809-be04-4a2d-8580-fb8a3921b086)

Best C value: 0.1


### Train the final model

**Objective**: Retrain the selected model using the combined training and validation datasets.

In [9]:
X_final_train = pd.concat([X_train, X_val])
y_final_train = pd.concat([y_train, y_val])

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        C=float(BEST_C),
        max_iter=1000,
        solver="lbfgs"
    ))
])

final_model.fit(X_final_train, y_final_train)

print("Final model training completed!")

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 13, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


### Evaluate the final model
**Objective:** Measure the final model's performance using the independent test dataset.

In [10]:
from sklearn.metrics import classification_report

test_predictions = final_model.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_f1 = f1_score(
    y_test,
    test_predictions
)

print("Test Accuracy:", round(test_accuracy, 4))
print("Test F1 Score:", round(test_f1, 4))

print(classification_report(
    y_test,
    test_predictions
))

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 14, Finished, Available, Finished, False)

Test Accuracy: 0.805
Test F1 Score: 0.7826
              precision    recall  f1-score   support

           0       0.81      0.83      0.82       545
           1       0.79      0.77      0.78       455

    accuracy                           0.81      1000
   macro avg       0.80      0.80      0.80      1000
weighted avg       0.80      0.81      0.80      1000



### Log the final model in MLflow

**Objective**: Log the final model, its selected parameter, and its test metrics.

In [11]:
# MODEL_NAME = "CustomerChurnModel"

with mlflow.start_run(
    run_name="CustomerChurn_Final_Model"
) as final_run:

    mlflow.log_param(
        "algorithm",
        "LogisticRegression"
    )

    mlflow.log_param(
        "regularization_C",
        float(BEST_C)
    )

    mlflow.log_param(
        "training_records",
        len(X_final_train)
    )

    mlflow.log_metric(
        "test_accuracy",
        test_accuracy
    )

    mlflow.log_metric(
        "test_f1_score",
        test_f1
    )

    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model"
    )

    BEST_MODEL_RUN_ID = final_run.info.run_id

print("Final model logged!")
print("Run ID:", BEST_MODEL_RUN_ID)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 15, Finished, Available, Finished, False)

🏃 View run CustomerChurn_Final_Model at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f/runs/8c919efd-0024-4581-8875-f476b1467d06
🧪 View experiment at: https://api.fabric.microsoft.com/v1/workspaces/41601757-cdc0-40a2-92fa-fb05e848aefc/mlflow/#/experiments/80d01eb9-849b-46fe-afc1-8363e6ee6c8f


Final model logged!
Run ID: 8c919efd-0024-4581-8875-f476b1467d06


### Register the final model

**Objective**: Register the final model in the MLflow model registry.

MODEL_NAME is the registered model's name in Microsoft Fabric MLflow. It is not the experiment name or the run ID.

In [12]:
MODEL_NAME = "CustomerChurnModelV2"
model_uri = f"runs:/{BEST_MODEL_RUN_ID}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

print("Model registered successfully!")
print("Model name:", MODEL_NAME)
print("Model version:", registered_model.version)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 16, Finished, Available, Finished, False)

Successfully registered model 'CustomerChurnModelV2'.
Created version '1' of model 'CustomerChurnModelV2'.


### Download the registered model

**Objective**: Download the registered model artifacts from MLflow.

In [13]:
MODEL_URI = (
    f"models:/{MODEL_NAME}/{registered_model.version}"
)

download_path = mlflow.artifacts.download_artifacts(
    artifact_uri=MODEL_URI
)

print("Downloaded model path:", download_path)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 18, Finished, Available, Finished, False)

Downloaded model path: /tmp/tmpvneto6tc/


StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 19, Finished, Available, Finished, False)

### Store model artifacts in the Lakehouse

**Objective**: Copy the downloaded model artifacts into the MLTrainingLakehouse Files area.

In [14]:
import os
import shutil

LAKEHOUSE_MODEL_PATH = (
    "/lakehouse/default/Files/Models/CustomerChurnModel"
)

os.makedirs(
    os.path.dirname(LAKEHOUSE_MODEL_PATH),
    exist_ok=True
)

shutil.copytree(
    download_path,
    LAKEHOUSE_MODEL_PATH,
    dirs_exist_ok=True
)

print("Model artifacts copied successfully!")

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 20, Finished, Available, Finished, False)

Model artifacts copied successfully!


### Load the model from Lakehouse

**Objective**: Load the model artifacts stored in Lakehouse Files.

In [15]:
loaded_model = mlflow.sklearn.load_model(
    LAKEHOUSE_MODEL_PATH
)

print("Model loaded successfully from Lakehouse!")

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 21, Finished, Available, Finished, False)

Model loaded successfully from Lakehouse!


### Predict churn for new customers

**Objective**: Use the saved model to predict churn for previously unseen customer records.

In [16]:
new_customers = pd.DataFrame({
    "tenure_months": [3, 60, 12, 48],
    "monthly_charges": [140.0, 65.0, 120.0, 75.0],
    "support_calls": [9, 1, 7, 2],
    "contract_type": [0, 2, 0, 1]
})

predictions = loaded_model.predict(new_customers)

probabilities = loaded_model.predict_proba(
    new_customers
)[:, 1]

prediction_results = new_customers.copy()

prediction_results["predicted_churn"] = predictions

prediction_results["churn_probability"] = (
    probabilities.round(4)
)

prediction_results["churn_status"] = np.where(
    predictions == 1,
    "Likely to Churn",
    "Not Likely to Churn"
)

display(prediction_results)

StatementMeta(, 5d498612-1322-4a3a-8e5d-644fc9c5fd98, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1e2b9a5-741b-47de-b4e8-9c72b64db466)